# Memory in the LangChain Agent

## Agent Without Memory

In [1]:
# import the necessary variables and tools
from lib.order_process_helper import CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT, get_order_status

In [2]:
model_name ="gpt-5-nano"
tools = [get_order_status]

In [ ]:
# define tools and agent with the system prompt and tools
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


agent = create_agent(
    model=model_name,
    system_prompt=CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT,
    tools=tools
)

In [4]:
# Inquire about the order status
from langchain.messages import HumanMessage
human_message = HumanMessage("我的訂單 A1024 現在送到哪裡了？什麼時候會到？")


In [5]:
# invoke the agent with the human message and get the response
# Invoke with the AgentState Scheme
response_1 = agent.invoke({"messages": [human_message]})

In [6]:
from pprint import pp

pp(response_1["messages"][-1].content)

('A1024 現在狀態：已出貨（Shipped）。\n'
 '預計送達日期：2023-07-25。\n'
 '\n'
 '目前系統僅顯示狀態與預計到達日期，未提供包裹的實時投遞地點資訊。如需更詳情，請提供追蹤號碼，或告訴我是否要我檢視最新追蹤狀態。')


In [7]:
# 詢問我上次問的訂單號碼
human_message_2 = HumanMessage("我上次問的訂單號碼是什麼？")
# invoke the agent with the human message and get the response
response_2 = agent.invoke({"messages": [human_message_2]})

In [ ]:
# print the response content
# No memory (stateless)
pp(response_2["messages"][-1].content)

'抱歉，我無法看到您上次對話中的訂單號碼。請提供您的訂單號碼（例如：ABC12345），我就能幫您查詢目前的狀態。若您不確定訂單號碼，也可以提供下單時使用的電子郵件、收件人姓名與下單日期，我可以協助找出相符的訂單。'


## Agent With Memory

In [3]:
# Add the short-term memory to the agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [5]:
# Create a new agent with memory
from langchain.agents import create_agent
agent_with_memory = create_agent(
    model=model_name,
    system_prompt=CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT,
    tools=tools,
    checkpointer=checkpointer
)

In [9]:
# Define a thread id to group the multi-tern questions and answers

import uuid
from pprint import pp


thread_id = str(uuid.uuid4())
pp(thread_id)

'0c89ffb8-e2d5-47b7-88d8-7e3d32148be8'


In [ ]:
# As the first question
from langchain.messages import HumanMessage
human_message_3 = HumanMessage("我的訂單 A1024 現在送到哪裡了？什麼時候會到？")

# Set the config for the invoke method to include the thread_id
# RunnableConfig? 
thread_config = {
    "configurable": {
        "thread_id": thread_id
    }
}
pp(thread_config)

{'configurable': {'thread_id': '0c89ffb8-e2d5-47b7-88d8-7e3d32148be8'}}


In [11]:
# The first round of invoking the agent with memory
response_3 = agent_with_memory.invoke({"messages": [human_message_3]}, config=thread_config)


In [12]:
pp(response_3["messages"][-1].content)

('您好。您的訂單 A1024 目前狀態為 已出貨，預計送達日期為 2023-07-25。系統目前沒有顯示具體的配送地址或追蹤號碼。\n'
 '\n'
 '如果您需要，我可以幫您查詢最新的追蹤資訊或協助與承運商聯繫，請告訴我您需要的協助方式。')


In [ ]:
# Show the AgentState of the agent with memory
# agent_with_memory.get_state() returns a StateSnapshot object
current_snapshot = agent_with_memory.get_state(thread_config).values["messages"]
for message in current_snapshot:
    pp(message)

HumanMessage(content='我的訂單 A1024 現在送到哪裡了？什麼時候會到？', additional_kwargs={}, response_metadata={}, id='d59e23c7-8274-4739-81cd-cbaee633732f')
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 362, 'total_tokens': 645, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8hmQCIXE6OJG0SvRBFJXdaa1uB6e', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc69d-3a03-7790-bb33-7c53b2641817-0', tool_calls=[{'name': 'get_order_status', 'args': {'order_id': 'A1024'}, 'id': 'call_BhERjfTYNRP0VsBivLDxcU90', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3

In [17]:
# Ask again
human_message_4 = HumanMessage("我上次問的訂單號碼是什麼？")
# invoke the agent with the human message and get the response
response_4 = agent_with_memory.invoke({"messages": [human_message_4]}, config=thread_config)

In [18]:
pp(response_4["messages"][-1].content)

'上次問的訂單號碼是 A1024。需要我幫你查詢該訂單的最新狀態嗎？'


In [19]:
# print the current AgentState
for message in agent_with_memory.get_state(thread_config).values["messages"]:
    pp(message)

HumanMessage(content='我的訂單 A1024 現在送到哪裡了？什麼時候會到？', additional_kwargs={}, response_metadata={}, id='d59e23c7-8274-4739-81cd-cbaee633732f')
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 362, 'total_tokens': 645, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8hmQCIXE6OJG0SvRBFJXdaa1uB6e', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc69d-3a03-7790-bb33-7c53b2641817-0', tool_calls=[{'name': 'get_order_status', 'args': {'order_id': 'A1024'}, 'id': 'call_BhERjfTYNRP0VsBivLDxcU90', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3